# ✦ LILY WAN 2.2 — Dual-T4 Image → Video Studio

**iPhone/Kaggle setup:** turn **Internet ON** and choose **GPU T4 x2**, then run the single code cell below.

The notebook uses Wan 2.2 TI2V-5B in ComfyUI's native low-VRAM path on GPU 0, optional MagCache acceleration, and RIFE frame interpolation on GPU 1. If RIFE or MagCache cannot initialize, generation still opens in fallback mode rather than failing the whole notebook.

Presets: **Turbo = 61 native frames / 12 diffusion steps**, **Normal = 81 / 16**, **Max = 121 / 20**. Turbo and Normal are finished to phone-friendly 24 fps; Max generates the full native sequence.


In [ ]:

import os, sys, subprocess, time, json, shutil, glob, uuid, random, socket, traceback
from pathlib import Path

# =========================
# LILY WAN 2.2 KAGGLE STUDIO
# =========================
# Designed for Kaggle T4 x2.
# GPU 0: Wan 2.2 TI2V-5B / ComfyUI
# GPU 1: RIFE frame interpolation when available
#
# Before running:
#   Kaggle -> Settings -> Accelerator -> GPU T4 x2
#   Kaggle -> Settings -> Internet -> On
#
# Then run this one cell.

ROOT = Path("/kaggle/working")
COMFY = ROOT / "ComfyUI"
OUTPUTS = ROOT / "lily_outputs"
CACHE = ROOT / "lily_model_cache"
RIFE = ROOT / "Practical-RIFE"
OUTPUTS.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None, check=True):
    print("›", " ".join(map(str, cmd)))
    return subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
        text=True,
    )

def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])

print("\n=== Hardware ===")
try:
    run(["nvidia-smi", "-L"])
except Exception:
    pass

# Core helpers/UI.
pip_install("gradio>=5.20,<6", "huggingface_hub>=0.29", "requests>=2.32", "gdown>=5.2", "scikit-video")

import requests
import gradio as gr
from PIL import Image
from huggingface_hub import hf_hub_download

# -------------------------
# 1) Install/update ComfyUI
# -------------------------
if not COMFY.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)])
else:
    # Keep an existing install usable; don't fail the whole notebook if update is unavailable.
    try:
        run(["git", "pull", "--ff-only"], cwd=COMFY)
    except Exception as e:
        print("ComfyUI update skipped:", e)

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)

# -------------------------
# 2) MagCache acceleration
# -------------------------
MAG_DIR = COMFY / "custom_nodes" / "ComfyUI-MagCache"
if not MAG_DIR.exists():
    try:
        run(["git", "clone", "--depth", "1", "https://github.com/Zehong-Ma/ComfyUI-MagCache.git", str(MAG_DIR)])
    except Exception as e:
        print("MagCache clone failed; baseline Wan will still work:", e)
else:
    try:
        run(["git", "pull", "--ff-only"], cwd=MAG_DIR)
    except Exception:
        pass

MAGCACHE_PATCHED = False
nodes_py = MAG_DIR / "nodes.py"
if nodes_py.exists():
    try:
        s = nodes_py.read_text()
        # Current upstream contains Wan2.2 ratios/forward logic but some revisions omit
        # wan2.2_ti2v_5B from the visible model_type list. Add it only when needed.
        needle = '"wan2.1_vace_14B"], {"default":'
        replacement = '"wan2.1_vace_14B", "wan2.2_ti2v_5B"], {"default":'
        if needle in s:
            s = s.replace(needle, replacement, 1)
            nodes_py.write_text(s)
            MAGCACHE_PATCHED = True
        elif '"wan2.2_ti2v_5B"], {"default":' in s:
            MAGCACHE_PATCHED = True
        req = MAG_DIR / "requirements.txt"
        if req.exists():
            run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
    except Exception as e:
        print("MagCache patch/setup warning:", e)

# -------------------------
# 3) Find/download model files
# -------------------------
MODEL_SPECS = [
    {
        "name": "wan2.2_ti2v_5B_fp16.safetensors",
        "repo": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
        "remote": "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
        "dest": COMFY / "models" / "diffusion_models",
    },
    {
        "name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
        "remote": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "dest": COMFY / "models" / "text_encoders",
    },
    {
        "name": "wan2.2_vae.safetensors",
        "repo": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
        "remote": "split_files/vae/wan2.2_vae.safetensors",
        "dest": COMFY / "models" / "vae",
    },
]

def find_attached_file(name):
    # If Lily later attaches a Kaggle Dataset containing these exact filenames,
    # startup becomes a symlink operation instead of a re-download.
    for p in Path("/kaggle/input").rglob(name):
        if p.is_file():
            return p
    return None

def ensure_model(spec):
    dest_dir = spec["dest"]
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / spec["name"]
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print("✓ model already present:", spec["name"])
        return dest

    attached = find_attached_file(spec["name"])
    if attached:
        print("✓ cache hit in /kaggle/input:", attached)
        if dest.exists() or dest.is_symlink():
            dest.unlink()
        dest.symlink_to(attached)
        return dest

    print("↓ downloading once for this session:", spec["name"])
    cached = hf_hub_download(
        repo_id=spec["repo"],
        filename=spec["remote"],
        cache_dir=str(CACHE / "hf"),
        local_dir=None,
    )
    if dest.exists() or dest.is_symlink():
        dest.unlink()
    dest.symlink_to(Path(cached))
    return dest

for spec in MODEL_SPECS:
    ensure_model(spec)

# -------------------------
# 4) Optional RIFE on GPU 1
# -------------------------
# RIFE 4.25.lite is intentionally optional. A failed RIFE install never prevents
# Wan generation; the app falls back to ffmpeg interpolation / direct encode.
RIFE_READY = False
try:
    if not RIFE.exists():
        run(["git", "clone", "--depth", "1", "https://github.com/hzwer/Practical-RIFE.git", str(RIFE)])
    else:
        try:
            run(["git", "pull", "--ff-only"], cwd=RIFE)
        except Exception:
            pass

    # Practical-RIFE still calls legacy NumPy aliases through scikit-video on some runtimes.
    rife_script = RIFE / "inference_video.py"
    if rife_script.exists():
        rs = rife_script.read_text()
        compat = 'if not hasattr(np, "float"):\n    np.float = float\nif not hasattr(np, "int"):\n    np.int = int\n'
        if compat not in rs and "import numpy as np" in rs:
            rs = rs.replace("import numpy as np\n", "import numpy as np\n" + compat, 1)
            rife_script.write_text(rs)

    train_log = RIFE / "train_log"
    if not (train_log / "flownet.pkl").exists():
        import gdown, zipfile
        model_archive = ROOT / "rife425lite"
        print("↓ downloading lightweight RIFE 4.25 model...")
        gdown.download(id="1zlKblGuKNatulJNFf5jdB-emp9AqGK05", output=str(model_archive), quiet=False)
        if model_archive.exists() and zipfile.is_zipfile(model_archive):
            extract_dir = ROOT / "rife_extract"
            if extract_dir.exists():
                shutil.rmtree(extract_dir)
            extract_dir.mkdir()
            with zipfile.ZipFile(model_archive) as z:
                z.extractall(extract_dir)
            candidates = list(extract_dir.rglob("flownet.pkl"))
            if candidates:
                src_dir = candidates[0].parent
                if train_log.exists():
                    shutil.rmtree(train_log)
                shutil.copytree(src_dir, train_log)
    RIFE_READY = (train_log / "flownet.pkl").exists()
except Exception as e:
    print("RIFE unavailable; safe fallback will be used:", e)
    RIFE_READY = False

if shutil.which("ffmpeg") is None:
    try:
        run(["apt-get", "update"], check=False)
        run(["apt-get", "install", "-y", "ffmpeg"], check=False)
    except Exception:
        pass

print("RIFE:", "READY ✓" if RIFE_READY else "fallback mode")

# -------------------------
# 5) Start ComfyUI on GPU 0
# -------------------------
COMFY_URL = "http://127.0.0.1:8188"
log_path = ROOT / "comfyui.log"

def comfy_alive():
    try:
        return requests.get(COMFY_URL + "/system_stats", timeout=2).ok
    except Exception:
        return False

if not comfy_alive():
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    # T4 = FP16 card. --lowvram lets ComfyUI aggressively offload around the 16 GB ceiling.
    cmd = [
        sys.executable, "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--disable-auto-launch",
        "--lowvram",
    ]
    log_f = open(log_path, "w")
    COMFY_PROC = subprocess.Popen(cmd, cwd=str(COMFY), env=env, stdout=log_f, stderr=subprocess.STDOUT)
    print("Starting ComfyUI...")
    for _ in range(180):
        if comfy_alive():
            break
        time.sleep(1)
    else:
        tail = ""
        try:
            tail = "\n".join(log_path.read_text(errors="ignore").splitlines()[-80:])
        except Exception:
            pass
        raise RuntimeError("ComfyUI did not start.\n\nLast log lines:\n" + tail)

print("✓ ComfyUI backend ready")

# Validate nodes before the user ever sees a Generate button.
obj = requests.get(COMFY_URL + "/object_info", timeout=30).json()
required_nodes = [
    "UNETLoader", "CLIPLoader", "VAELoader", "LoadImage",
    "CLIPTextEncode", "Wan22ImageToVideoLatent", "ModelSamplingSD3",
    "KSampler", "VAEDecode", "CreateVideo", "SaveVideo",
]
missing = [n for n in required_nodes if n not in obj]
if missing:
    raise RuntimeError("Missing required ComfyUI nodes: " + ", ".join(missing))

MAGCACHE_READY = "MagCache" in obj
if MAGCACHE_READY:
    try:
        choices = obj["MagCache"]["input"]["required"]["model_type"][0]
        MAGCACHE_READY = "wan2.2_ti2v_5B" in choices
    except Exception:
        pass
print("MagCache:", "READY ✓" if MAGCACHE_READY else "baseline mode")

# -------------------------
# 6) ComfyUI API client
# -------------------------
NEG_DEFAULT = (
    "overexposed, oversaturated, static frame, blurry details, subtitles, text, watermark, "
    "worst quality, low quality, jpeg artifacts, malformed anatomy, deformed hands, extra fingers, "
    "fused fingers, duplicate limbs, frozen motion, cluttered background"
)

PRESETS = {
    # Frame counts are 4k+1, as Wan expects.
    # Lower fps makes each native sequence ~5 seconds before interpolation.
    "⚡ Turbo":  {"frames": 61,  "steps": 12, "source_fps": 12.0, "cfg": 5.0},
    "✨ Normal": {"frames": 81,  "steps": 16, "source_fps": 16.0, "cfg": 5.0},
    "👑 Max":    {"frames": 121, "steps": 20, "source_fps": 24.0, "cfg": 5.0},
}
SIZES = {
    "Landscape 16:9": (1280, 704),
    "Portrait 9:16": (704, 1280),
}

def upload_to_comfy(pil_image):
    import io
    buf = io.BytesIO()
    pil_image.convert("RGB").save(buf, format="PNG")
    buf.seek(0)
    name = f"lily_{uuid.uuid4().hex}.png"
    r = requests.post(
        COMFY_URL + "/upload/image",
        files={"image": (name, buf.getvalue(), "image/png")},
        data={"type": "input", "overwrite": "true"},
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    # ComfyUI returns name/subfolder/type.
    sub = data.get("subfolder", "")
    return f"{sub}/{data['name']}" if sub else data["name"]

def build_workflow(image_name, prompt, negative, preset_name, orientation, seed, use_magcache):
    p = PRESETS[preset_name]
    width, height = SIZES[orientation]

    wf = {
        "1": {
            "class_type": "UNETLoader",
            "inputs": {
                "unet_name": "wan2.2_ti2v_5B_fp16.safetensors",
                "weight_dtype": "default",
            },
        },
        "2": {
            "class_type": "CLIPLoader",
            "inputs": {
                "clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
                "type": "wan",
                "device": "default",
            },
        },
        "3": {
            "class_type": "VAELoader",
            "inputs": {"vae_name": "wan2.2_vae.safetensors"},
        },
        "4": {
            "class_type": "LoadImage",
            "inputs": {"image": image_name},
        },
        "5": {
            "class_type": "CLIPTextEncode",
            "inputs": {"text": prompt, "clip": ["2", 0]},
        },
        "6": {
            "class_type": "CLIPTextEncode",
            "inputs": {"text": negative, "clip": ["2", 0]},
        },
        "7": {
            "class_type": "Wan22ImageToVideoLatent",
            "inputs": {
                "width": width,
                "height": height,
                "length": p["frames"],
                "batch_size": 1,
                "vae": ["3", 0],
                "start_image": ["4", 0],
            },
        },
    }

    model_node = "1"
    if use_magcache and MAGCACHE_READY:
        wf["8"] = {
            "class_type": "MagCache",
            "inputs": {
                "model": ["1", 0],
                "model_type": "wan2.2_ti2v_5B",
                "magcache_thresh": 0.06,
                "retention_ratio": 0.20,
                "magcache_K": 2,
                "start_step": 0,
                "end_step": -1,
            },
        }
        model_node = "8"

    wf["9"] = {
        "class_type": "ModelSamplingSD3",
        "inputs": {"model": [model_node, 0], "shift": 8.0},
    }
    wf["10"] = {
        "class_type": "KSampler",
        "inputs": {
            "model": ["9", 0],
            "positive": ["5", 0],
            "negative": ["6", 0],
            "latent_image": ["7", 0],
            "seed": int(seed),
            "steps": p["steps"],
            "cfg": p["cfg"],
            "sampler_name": "uni_pc",
            "scheduler": "simple",
            "denoise": 1.0,
        },
    }
    wf["11"] = {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["10", 0], "vae": ["3", 0]},
    }
    wf["12"] = {
        "class_type": "CreateVideo",
        "inputs": {"images": ["11", 0], "fps": p["source_fps"]},
    }
    wf["13"] = {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["12", 0],
            "filename_prefix": "video/LILY_WAN22",
            "format": "auto",
            "codec": "auto",
        },
    }
    return wf

def queue_workflow(wf):
    r = requests.post(COMFY_URL + "/prompt", json={"prompt": wf}, timeout=60)
    if not r.ok:
        raise RuntimeError(f"ComfyUI rejected workflow ({r.status_code}): {r.text[:2500]}")
    return r.json()["prompt_id"]

def wait_for_prompt(prompt_id, timeout=3600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        r = requests.get(COMFY_URL + f"/history/{prompt_id}", timeout=30)
        if r.ok:
            hist = r.json()
            if prompt_id in hist:
                item = hist[prompt_id]
                status = item.get("status", {})
                if status.get("status_str") == "error":
                    msgs = status.get("messages", [])
                    raise RuntimeError("ComfyUI generation error:\n" + json.dumps(msgs[-5:], indent=2)[:5000])
                if status.get("completed", False):
                    return item
        time.sleep(2)
    raise TimeoutError("Generation exceeded the notebook timeout.")

def newest_generated_mp4(after_time):
    files = []
    for ext in ("*.mp4", "*.mov", "*.mkv", "*.webm"):
        files.extend((COMFY / "output").rglob(ext))
    files = [p for p in files if p.stat().st_mtime >= after_time - 2]
    if not files:
        raise FileNotFoundError("Generation completed but no saved video was found.")
    return max(files, key=lambda p: p.stat().st_mtime)

def normalize_for_iphone(src, dst, fps=24):
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-vf", f"fps={fps}",
        "-an",
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(dst),
    ]
    run(cmd, check=True)
    return dst

def rife_interpolate(src, dst, source_fps):
    if not RIFE_READY:
        raise RuntimeError("RIFE not ready")
    # RIFE on physical GPU 1. If Kaggle unexpectedly gave only one GPU, use GPU 0.
    try:
        import torch
        gpu_count = torch.cuda.device_count()
    except Exception:
        gpu_count = 1
    physical_gpu = "1" if gpu_count >= 2 else "0"
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = physical_gpu

    # 2x frames at 2x source fps preserves the original ~5 s duration.
    out_fps = max(2, int(round(float(source_fps) * 2)))
    cmd = [
        sys.executable, "inference_video.py",
        "--video", str(src),
        "--output", str(dst),
        "--model", str(RIFE / "train_log"),
        "--multi", "2",
        "--fps", str(out_fps),
        "--fp16",
    ]
    run(cmd, cwd=RIFE, env=env, check=True)
    if not dst.exists() or dst.stat().st_size < 10_000:
        raise RuntimeError("RIFE produced no valid output")
    return dst

def fallback_interpolate(src, dst, fps=24):
    # Motion-compensated interpolation with ffmpeg. Slower than simple frame duplication,
    # but no extra neural model is required.
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-vf", f"minterpolate=fps={fps}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1",
        "-an",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "19",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(dst),
    ]
    run(cmd, check=True)
    return dst

generation_lock = __import__("threading").Lock()

def generate(image, prompt, negative, preset_name, orientation, seed, randomize_seed, magcache, interpolation):
    if image is None:
        raise gr.Error("Add an image first.")
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Write a motion prompt.")

    with generation_lock:
        if randomize_seed:
            seed = random.randint(0, 2**63 - 1)
        seed = int(seed)

        try:
            image_name = upload_to_comfy(image)
            wf = build_workflow(
                image_name=image_name,
                prompt=prompt,
                negative=(negative or NEG_DEFAULT),
                preset_name=preset_name,
                orientation=orientation,
                seed=seed,
                use_magcache=bool(magcache),
            )

            started = time.time()
            prompt_id = queue_workflow(wf)
            wait_for_prompt(prompt_id)
            raw = newest_generated_mp4(started)

            final = OUTPUTS / f"lily_{preset_name.split()[-1].lower()}_{seed}_{uuid.uuid4().hex[:6]}.mp4"

            if interpolation == "RIFE 2× → 24fps" and preset_name != "👑 Max":
                try:
                    rife_tmp = OUTPUTS / f"_rife_{uuid.uuid4().hex[:8]}.mp4"
                    rife_interpolate(raw, rife_tmp, PRESETS[preset_name]["source_fps"])
                    normalize_for_iphone(rife_tmp, final, 24)
                    try:
                        rife_tmp.unlink()
                    except Exception:
                        pass
                    mode = "RIFE on GPU 1"
                except Exception as e:
                    print("RIFE failed; using ffmpeg fallback:", e)
                    fallback_interpolate(raw, final, 24)
                    mode = "ffmpeg interpolation fallback"
            elif interpolation == "Smooth 24fps (no RIFE)" and preset_name != "👑 Max":
                fallback_interpolate(raw, final, 24)
                mode = "ffmpeg interpolation"
            else:
                normalize_for_iphone(raw, final, 24)
                mode = "direct 24fps encode"

            elapsed = time.time() - started
            msg = (
                f"✅ Done • {elapsed/60:.1f} min • seed {seed} • "
                f"{PRESETS[preset_name]['frames']} native frames / {PRESETS[preset_name]['steps']} steps • {mode}"
            )
            if magcache and not MAGCACHE_READY:
                msg += " • MagCache unavailable, baseline model used"
            return str(final), msg, seed
        except Exception as e:
            # Surface the useful Comfy log tail instead of an opaque Gradio failure.
            print(traceback.format_exc())
            tail = ""
            try:
                tail = "\n".join(log_path.read_text(errors="ignore").splitlines()[-35:])
            except Exception:
                pass
            raise gr.Error(f"{type(e).__name__}: {e}\n\nBackend tail:\n{tail[-3500:]}")

# -------------------------
# 7) iPhone-friendly UI
# -------------------------
css = """
.gradio-container {max-width: 920px !important; margin: auto !important;}
#hero {text-align:center; padding: 6px 0 2px 0;}
#status textarea {font-size: 14px !important;}
"""

with gr.Blocks(css=css, title="Lily Wan 2.2 Studio") as demo:
    gr.Markdown(
        "# ✦ Lily Wan 2.2 Studio\n"
        "Image → video on your Kaggle GPUs. **Turbo** is the everyday mode; **Max** is the quality mode.",
        elem_id="hero",
    )
    with gr.Row():
        with gr.Column(scale=1):
            image = gr.Image(label="Starting image", type="pil", sources=["upload", "clipboard"])
            prompt = gr.Textbox(
                label="Motion prompt",
                lines=5,
                placeholder="Describe what moves, how the subject moves, and how the camera moves…",
            )
            negative = gr.Textbox(label="Negative prompt", value=NEG_DEFAULT, lines=2)
        with gr.Column(scale=1):
            preset = gr.Radio(list(PRESETS), value="⚡ Turbo", label="Quality / speed")
            orientation = gr.Radio(list(SIZES), value="Portrait 9:16", label="Format")
            magcache = gr.Checkbox(value=True, label="MagCache acceleration")
            interp_default = "RIFE 2× → 24fps" if RIFE_READY else "Smooth 24fps (no RIFE)"
            interpolation = gr.Radio(
                ["RIFE 2× → 24fps", "Smooth 24fps (no RIFE)", "Direct 24fps"],
                value=interp_default,
                label="Finish",
            )
            seed = gr.Number(value=42, precision=0, label="Seed")
            randomize = gr.Checkbox(value=True, label="Randomize seed")
            go = gr.Button("✦ GENERATE VIDEO", variant="primary", size="lg")
    out = gr.Video(label="Result", autoplay=True)
    status = gr.Textbox(label="Status", interactive=False, elem_id="status")
    actual_seed = gr.Number(label="Used seed", precision=0, interactive=False)

    gr.Markdown(
        f"Backend: **Wan 2.2 TI2V-5B FP16** • GPU 0 generation • "
        f"GPU 1 RIFE: **{'ready' if RIFE_READY else 'fallback'}** • "
        f"MagCache: **{'ready' if MAGCACHE_READY else 'baseline'}**"
    )

    go.click(
        fn=generate,
        inputs=[image, prompt, negative, preset, orientation, seed, randomize, magcache, interpolation],
        outputs=[out, status, actual_seed],
        concurrency_limit=1,
    )

demo.queue(max_size=20)
print("\nLaunching Lily Studio. Open the public Gradio link printed below on your iPhone.\n")
demo.launch(
    share=True,
    server_name="0.0.0.0",
    show_error=True,
    allowed_paths=[str(OUTPUTS), str(COMFY / "output")],
)
